# Importando Bibliotecas

In [1]:
# Libs Iniciais
import pandas as pd
import requests
import re

#Libs para tratamento de Dados adicionais
import pycountry

#Libs para carregar dados
from sqlalchemy import create_engine
from sqlalchemy.sql import text
from typing import Optional
from dotenv import load_dotenv

# Funções Auxiliadores

In [2]:
"""
Função responsável pela correção do nome do país
Ex: Korea, republic of -> republic of Korea 
"""
def arranged_country(country:str| None) -> str | None:
    if pd.notnull(country):
        splitted_country = country.split(',')
    else:
        return country

    return " ".join(splitted_country[::-1])

"""
Função que será responsável por quaisquer buscas na base do Clinical Trials por API,
sendo assim, basta apenas inserir a url. 
"""
def fetchAPIClinicalTrials(*,url:str, next_page:bool=True, token:Optional[str]=None, clinical_trials:list[pd.DataFrame]=None) -> pd.DataFrame:
    if clinical_trials is None:
        clinical_trials = []
    
    while next_page:
        response = requests.get(url).json()
        studies = response.get('studies', [])

        if studies:
            clinical_trials.append(pd.json_normalize(studies))
        
        token = response.get('nextPageToken')

        if token:
            if 'pageToken' not in url:
                url += f"&pageToken={token}"
            else:
                url = re.sub(r'pageToken=[^&]+', f'pageToken={token}', url)
        else:
            next_page = False
    
    df:pd.DataFrame = pd.concat(clinical_trials, ignore_index=True)
    return df

# Coletando os dados por Requisição com iteração

In [3]:

url ="https://clinicaltrials.gov/api/v2/studies?fields=NCTId%7CCondition%7CStartDate%7CStudyFirstPostDate%7CLeadSponsor%7CCollaborator%7CStudyType%7CPhase%7CEnrollmentCount%7CInterventionType%7CInterventionName%7CLocationCountry%7COfficialTitle%7CLeadSponsorClass%7COverallStatus&pageSize=1000"

df = fetchAPIClinicalTrials(url=url)

print(df.shape[0])

573677


# Fazendo o Tratamento de dados

In [4]:
map_columns = {
    'protocolSection.identificationModule.nctId': 'NCT Number',
    'protocolSection.identificationModule.officialTitle': 'Study Title',
    'protocolSection.statusModule.startDateStruct.date': 'Start Date',
    'protocolSection.statusModule.studyFirstPostDateStruct.date': 'First Posted',
    'protocolSection.sponsorCollaboratorsModule.leadSponsor.name': 'Sponsor',
    'protocolSection.sponsorCollaboratorsModule.leadSponsor.class': 'Funder Type',
    'protocolSection.conditionsModule.conditions': 'Conditions',
    'protocolSection.designModule.studyType': 'Study Type',
    'protocolSection.designModule.phases': 'Phases',
    'protocolSection.designModule.enrollmentInfo.count': 'Enrollment',
    'protocolSection.armsInterventionsModule.interventions': 'Intervention/Intervention Type',
    'protocolSection.contactsLocationsModule.locations': 'Country',
    'protocolSection.sponsorCollaboratorsModule.collaborators': 'Collaborators',
    'protocolSection.statusModule.overallStatus': 'Study Status'
}

df = df.rename(columns=map_columns)


## Tratando Intervenções

In [5]:
df_interventions = df[["NCT Number", 'Intervention/Intervention Type']].copy()

df_interventions = df_interventions.explode('Intervention/Intervention Type')

df_interventions['Intervention Type'] = df_interventions['Intervention/Intervention Type'].apply(lambda x: x.get('type') if pd.notnull(x) else x)
df_interventions['Intervention'] = df_interventions['Intervention/Intervention Type'].apply(lambda x: x.get('name') if pd.notnull(x) else x)

df_interventions = df_interventions.drop(columns='Intervention/Intervention Type')

df_interventions.drop_duplicates(inplace=True)

## Tratando País

In [6]:
df_locations = df[['NCT Number', 'Country']].copy()

df_locations = df_locations.explode('Country')

df_locations["Country"] = df_locations['Country'].apply(lambda x: x.get('country') if pd.notnull(x) else x)

df_locations['Country'] = df_locations['Country'].apply(arranged_country)



countries = {}
for country in pycountry.countries:
    name = country.name
    if name.__contains__('Türkiye'):
        countries['Turkey'] = country.alpha_3
    elif name.__contains__('Cabo Verde'):
        countries["Cape Verde"] = country.alpha_3
    elif name.__contains__('Venezuela'):
        countries["Venezuela"] = country.alpha_3
    elif name.__contains__('Bolivia'):
        countries["Bolivia"] = country.alpha_3
    elif name.__contains__('Tanzania'):
        countries["Tanzania"] = country.alpha_3
    elif name.__contains__('Viet Nam'):
        countries['Vietnam'] = country.alpha_3
    elif name.__contains__('Taiwan'):
        countries['Taiwan'] = country.alpha_3
    elif name.__contains__(','):
        splitted_name = name.split(',')
        final_name = splitted_name[::-1]
        countries[" ".join(final_name)] = country.alpha_3
    else:
        countries[country.name] = country.alpha_3


df_locations['Code cca3'] = [countries.get(country, 'Unknown code') for country in df_locations.Country]
df_locations.drop_duplicates(inplace=True)

## Tratando Colaboradores

In [7]:
df_collaborators = df[['NCT Number', 'Collaborators']].copy()

df_collaborators = df_collaborators.explode('Collaborators')

df_collaborators['Collaborators'] = df_collaborators['Collaborators'].apply(lambda x: x.get('name') if pd.notnull(x) else x)

df_collaborators.drop_duplicates(inplace=True)

## Expandindo Coluna Patologias

In [8]:
df_conditions = df[['NCT Number', 'Conditions']].copy()
df_conditions = df_conditions.explode('Conditions')

df_conditions.drop_duplicates(inplace=True)

## Tabela Patrocinadores

In [9]:
df_sponsor = df[['NCT Number', 'Sponsor']].copy()

## Tabela Remanescente

### Coletando os Dados

In [10]:
list_of_columns = [
    'NCT Number', 'Study Title', 'Study Status',
    'Start Date', 'First Posted',
    'Funder Type', 'Study Type', 'Enrollment',
    'Phases',
]

In [11]:
df_leftover = df[list_of_columns].copy()
df_leftover = df_leftover.astype({'First Posted': "datetime64[us]"})

In [12]:
df_leftover = df_leftover.explode('Phases')

### Adicionando campo de Link

In [13]:
nct_link = 'https://clinicaltrials.gov/study/'

df_leftover['Study URL'] = df_leftover['NCT Number'].apply(lambda x: nct_link + x)

df_leftover.drop_duplicates(inplace=True)

## Preenchendo Dados Nulos/ Not A Number

In [14]:
df_leftover.fillna('Not Mentioned', inplace=True)
df_interventions.fillna('Not Mentioned', inplace=True)
df_conditions.fillna('Not Mentioned', inplace=True)
df_collaborators.fillna('Not Mentioned', inplace=True)
df_locations.fillna('Not Mentioned', inplace=True)
df_sponsor.fillna('Not Mentioned', inplace=True)

/tmp/ipykernel_2379/1896033948.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Not Mentioned' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_leftover.fillna('Not Mentioned', inplace=True)


# Inserindo os Dados no Postgre

## Configurando os dados

In [15]:
import os
load_dotenv()

url = os.getenv("URL")

engine = create_engine(
    url, 
    pool_pre_ping=True, 
)

## Deletando a Materialized View à Evitar Exceções

In [16]:
with engine.connect() as conn:
    conn.execute(text('DROP MATERIALIZED VIEW IF EXISTS clinical_trials_mv CASCADE;'))
    conn.commit()

## Enviando as Tabelas

In [17]:
df_leftover.to_sql('leftover_table', con=engine, if_exists='replace', index=False, chunksize=50_000)

11472

In [18]:
df_collaborators.to_sql('collaborators_table', con=engine,if_exists='replace', index=False, chunksize=50_000)

14935

In [19]:
df_locations.to_sql('locations_table', con=engine,if_exists='replace', index=False, chunksize=50_000)


16063

In [20]:
df_conditions.to_sql('conditions_table', con=engine,if_exists='replace', index=False, chunksize=50_000)

20722

In [21]:
df_interventions.to_sql('interventions_table', con=engine,if_exists='replace', index=False, chunksize=50_000)

20332

In [22]:
df_sponsor.to_sql('sponsor_table', con=engine,if_exists='replace', index=False, chunksize=50_000)

11677

## Criando a Materialized View

In [23]:
with engine.connect() as conn:
    conn.execute(text("""
    CREATE MATERIALIZED VIEW clinical_trials_mv AS 
	SELECT subquery."NCT Number",
	   subquery."Study URL",
       subquery."Study Title",
       subquery."Study Status",
       subquery."Funder Type",
       subquery."Sponsor",
       subquery."Intervention",
       subquery."Intervention Type",
       subquery."Enrollment",
       subquery."Conditions",
       subquery."Phases",
       subquery."Collaborators",
       subquery."First Posted",
       subquery."Start Date",
	subquery."Country",
       subquery."Code cca3",
       SUM(subquery.count_distinct) AS total_distinct
FROM
  (SELECT lt."NCT Number",
		  lt."Study URL",
          lt."Study Title",
          lt."Study Status",
          s."Sponsor",
          lt."Funder Type",
          i."Intervention",
          i."Intervention Type",
          lt."Enrollment",
          c."Conditions",
          lt."First Posted",
          lt."Start Date",
          col."Collaborators",
          lt."Phases",
	   l."Country",
          l."Code cca3",
          COUNT(DISTINCT lt."NCT Number") AS count_distinct
   FROM leftover_table lt
   INNER JOIN sponsor_table s ON lt."NCT Number" = s."NCT Number"
   INNER JOIN collaborators_table col ON lt."NCT Number" = col."NCT Number"
   INNER JOIN conditions_table c ON lt."NCT Number" = c."NCT Number"
   INNER JOIN interventions_table i ON lt."NCT Number" = i."NCT Number"
  -- INNER JOIN leftover_table lt ON sd."NCT Number" = lt."NCT Number"
   INNER JOIN locations_table l ON lt."NCT Number" = l."NCT Number" -- INNER JOIN
 --     study_design_table f ON sd."NCT Number" = f."NCT Number"

   GROUP BY lt."NCT Number",
	     lt."Study URL",
            lt."Study Title",
            lt."Study Status",
            s."Sponsor",
            lt."Funder Type",
            i."Intervention",
            i."Intervention Type",
            lt."Enrollment",
            c."Conditions",
            lt."First Posted",
            lt."Start Date",
            col."Collaborators",
            lt."Phases",
            l."Country",
            l."Code cca3"
  -- HAVING l."Country" = 'Brazil'
  ) AS subquery
GROUP BY subquery."NCT Number",
	  subquery."Study URL",
         subquery."Study Title",
         subquery."Study Status",
         subquery."Sponsor",
         subquery."Funder Type",
         subquery."Intervention",
         subquery."Intervention Type",
         subquery."Enrollment",
         subquery."Conditions",
         subquery."Phases",
         subquery."Collaborators",
         subquery."First Posted",
         subquery."Start Date",
         subquery."Country",
         subquery."Code cca3";
       """))
    conn.commit()